# toolforge on a GPU (Colab / Kaggle)

QLoRA SFT -> DPO -> BFCL-style eval for Qwen2.5-0.5B-Instruct (or 1.5B) on the cleaned glaive function-calling set.

**Status: untested.** This notebook was written alongside the CPU run reported in the README but has not been executed by the author (no GPU was available). Runtime: GPU (T4 works; use `model.dtype=float16` there).

In [ ]:
!git clone https://github.com/fernandogarzaaa/toolforge.git
%cd toolforge
!pip install -q -e ".[train,qlora,export]"

Optional: track to Weights & Biases instead of the local MLflow store (`tracking.backend=wandb`).

In [ ]:
CONFIG = "configs/gpu_qlora.yaml"
EXTRA = ""  # e.g. "-o model.dtype=float16" on a T4, or "-o model.name_or_path=Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
!toolforge data -c $CONFIG $EXTRA

In [ ]:
!toolforge sft -c $CONFIG $EXTRA   # re-run this cell after a disconnect: it resumes from the last checkpoint

In [ ]:
!toolforge dpo -c $CONFIG $EXTRA

In [ ]:
!toolforge eval -c $CONFIG $EXTRA
print(open("results/RESULTS.md").read())

Merge the adapters and convert to GGUF with llama.cpp's converter:

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!LLAMA_CPP_DIR=$PWD/llama.cpp toolforge export -c $CONFIG $EXTRA -o export.quantize=[]